# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring

**Task Type:** Ranking / Scoring (using probability of decline as the ranking score)

**Why:** 
The business problem we are solving is sorting a large list of content pages to prioritize which ones an editor/reviewer should check first. 
- In our dataset, **54.2% of the pages are declining** in search impressions (meaning over 16,000 pages out of 30,000). A binary classification system that outputs a flat "yes/no" label is not sufficient because a human reviewer has limited time and cannot review thousands of flagged pages.
- A clustering algorithm (unsupervised) would group pages but would not order them by urgency or need.
- Therefore, we frame this as a **ranking / scoring** problem. We want to predict a continuous priority score (specifically, the probability that a page is declining, potentially weighted by the page's search impressions or traffic value) and sort the pages in descending order. This ranked queue directly supports the editor's decision of which page to open and refresh first.


In [1]:
# Check the distribution of our primary target candidates in the raw data
import pandas as pd
df_raw = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
total_pages = len(df_raw)
declining_pages = (df_raw['trend_direction'] == 'down').sum()
print(f"Total content pages: {total_pages:,}")
print(f"Declining content pages (trend_direction == 'down'): {declining_pages:,} ({declining_pages / total_pages:.1%})")


Total content pages: 30,000
Declining content pages (trend_direction == 'down'): 16,262 (54.2%)


## 2. Target or proxy

**What we predict (Target):**
We predict `is_declining_label`, which is a binary indicator (1 if the page's search impressions decline by more than 20% compared to the prior period, 0 otherwise).

**Where the label comes from (Observed or Rule):**
This label is an **observed outcome** computed directly from search performance data. Specifically, Google Search Console logs the impressions for each page over two subsequent 30-day windows:
1. `impressions_last_30d` (the most recent 30 days)
2. `impressions_prev_30d` (days 31–60 back)
The percentage change is calculated as:
`trend_pct = (impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100`
If `trend_pct < -20`, then `trend_direction` is marked as `'down'` and `is_declining_label = 1`. Otherwise, it is `0`.

Since this label is computed from actual, observed clicks and search impressions recorded after the feature measurement window, it represents a real-world outcome. It is NOT defined by someone's manual rule (like a product-level `health_score` or a manual tag).

**Proxy Nature:**
The true target of interest is "which pages need a content refresh to recover their value". Since "needs refresh" is subjective and not directly logged in databases, we use the observed search impression decline (`is_declining_label`) as a highly correlated **proxy** for the page's loss of search visibility.


In [2]:
# Verify that the target label is observed and has no trivial leakages
# Print the relationship between trend_direction and the label we will sketch
trend_counts = df_raw['trend_direction'].value_counts()
print("Distribution of trend_direction:")
for direction, count in trend_counts.items():
    print(f"  {direction}: {count:,} ({count / len(df_raw):.1%})")


Distribution of trend_direction:
  down: 16,262 (54.2%)
  stable: 5,962 (19.9%)
  up: 4,388 (14.6%)
  new: 2,236 (7.5%)
  flat: 1,152 (3.8%)


## 3. Success metric

**Primary Success Metric:** **Precision@K (specifically Precision@50)**

**Why this metric:**
A human content reviewer only has a fixed amount of time each week and can only review a limited number of pages, e.g., $K = 50$ pages. 
- If our system ranks pages poorly, the top 50 slots will contain healthy pages that do not need updates. This results in **false positives**, which cost valuable reviewer hours (wasted reading pages that are fine) and cause **false negatives** (actually declining pages are missed and continue to lose search traffic).
- Precision@50 measures the percentage of the top 50 recommended pages that are *actually* declining. Maximizing Precision@50 directly minimizes wasted human labor.

**What number means "good":**
- **Baseline Rule:** Sorting pages by a hand-written rule or a simple heuristic (e.g. sorting by raw impressions or position) achieves a Precision@50 of only **0.240** (24%) on client-holdout splits.
- **Target "Good" Threshold:** The starter Random Forest model in the pipeline achieves a Precision@50 of **0.740** (74%).
- Therefore, a Precision@50 of **0.70 or higher (70%+)** is our benchmark for "good." This means that more than 7 out of 10 pages recommended to the reviewer are relevant, representing a 3x improvement over simple baseline heuristics.


In [3]:
# Compute a simple baseline rule: what if we prioritize by raw impressions_90d?
# Let's see what proportion of the top 50 pages sorted by impressions_90d are actually declining.
df_sorted_impressions = df_raw.sort_values(by='impressions_90d', ascending=False)
top_50_baseline = df_sorted_impressions.head(50)
precision_at_50 = (top_50_baseline['trend_direction'] == 'down').mean()
print(f"Precision@50 of a simple 'highest impressions' baseline rule: {precision_at_50:.1%}")


Precision@50 of a simple 'highest impressions' baseline rule: 42.0%


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** **One content item (page) for a specific client, aggregated over a trailing 90-day window.**

Each row in the dataset represents a single unique content page (identified by `content_id`) for a specific client website (`client_id`). The features are search console performance metrics (impressions, clicks, position) and analytics engagement metrics (pageviews, scroll events, engaged sessions) aggregated over a 90-day window.

Below we load the starter CSV file, show its shape, display the first few rows with key columns, and sketch the target column (`is_declining_label`).


In [4]:
# Load the dataset and show the unit of analysis as a real dataframe
import pandas as pd
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Verify the shape
print(f"Dataframe contains {df.shape[0]:,} rows and {df.shape[1]} columns.")
print(f"Number of unique content pages: {df['content_id'].nunique():,}")
print(f"Number of unique clients: {df['client_id'].nunique():,}")

# Sketch the target column as 'is_declining_label'
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Display a subset of columns for the first 5 rows to visualize the unit of analysis
cols_to_show = [
    'content_id', 
    'client_id', 
    'impressions_90d', 
    'clicks_90d', 
    'avg_position', 
    'ctr',
    'word_count',
    'trend_direction',
    'is_declining_label'
]
df[cols_to_show].head()


Dataframe contains 30,000 rows and 44 columns.
Number of unique content pages: 30,000
Number of unique clients: 32


,content_id,client_id,impressions_90d,clicks_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,10.6,0.76,3221.0,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,20.3,0.05,2481.0,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,3515.0,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,6.2,0.49,NaN,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,44.0,0.13,2803.0,down,1


## 5. Why ML beats a fixed rule here

**Why a plain rule (if-statement) is not enough:**
1. **High Dimensionality and Shifting Signals:** A page's decay is determined by multiple interacting factors. For example, a high-volume page with a slight rank drop might be losing more absolute traffic than a low-volume page with a large rank drop. A low Click-Through Rate (CTR) is normal for deep rankings (e.g. position 20), but indicates a severe content mismatch if the page is ranked in the top 3. Hardcoding these non-linear thresholds and interactions (e.g. `if impressions > X and position < Y and ctr < Z`) is highly error-prone and quickly becomes unmaintainable.
2. **Client Heterogeneity:** FlyRank services 32 different clients, ranging from small niche blogs to large enterprise sites. A fixed traffic threshold (e.g., "declining if clicks drop by 50") might trigger thousands of false alerts for a volatile small client, while failing to trigger for a large client where a 5% drop represents massive business value. ML models can learn relative patterns and incorporate client-specific distributions (e.g., using log-transformed features and client groupings) rather than hardcoded global constants.
3. **Continuous Probability Scoring:** A hand-written rule typically produces binary flags (e.g. "needs refresh" vs "does not"). It cannot easily produce a calibrated, continuous probability score that allows us to sort the pages. Machine learning models naturally output probabilities, enabling us to rank the entire catalog and always present the absolute top $K$ highest-probability candidates to the reviewer.


In [5]:
# Illustrate client heterogeneity: show how median impressions and click rates vary widely across clients
client_stats = df.groupby('client_id').agg(
    total_pages=('content_id', 'count'),
    median_impressions_90d=('impressions_90d', 'median'),
    median_clicks_90d=('clicks_90d', 'median'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()

print("Traffic and page distributions across a sample of 5 clients:")
print(client_stats.head(5).to_string(index=False))


Traffic and page distributions across a sample of 5 clients:
        client_id  total_pages  median_impressions_90d  median_clicks_90d  decline_rate
client_02d20bbd7e           38                    35.5                0.0      0.421053
client_0b918943df           35                    25.0                0.0      0.485714
client_19581e27de         7008                  1691.5                2.0      0.490154
client_1a6562590e            3                   342.0                0.0      0.000000
client_25fc0e7096          476                     3.0                0.0      0.000000


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
